# Notebook de démonstration — Applications IoT en élevage laitier
## Objectif 3 du SOW — Tâche 3.1

Ce notebook illustre, de bout en bout, l'exploitation de données IoT (capteurs accélérométriques
IceTag et capteurs environnementaux HOBO) pour le suivi de vaches laitières.

**Parcours :**
1. Chargement et prétraitement des données capteurs
2. Exécution du pipeline de détection d'anomalies comportementales (boiterie)
3. Visualisations environnement × comportement
4. Guide d'interprétation et limites

**Données :** 4 essais McGill (Fall 2019, Summer 2019, Winter 2019, Fall 2021).

*Ce notebook réutilise les résultats produits aux Objectifs 1 et 2 ; il sert de vitrine
pédagogique reproductible.*

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
O1 = PROJECT / 'reports' / 'objective1_pipeline_icetag'
O2 = PROJECT / 'reports' / 'objective2_environnement'
OUT = PROJECT / 'reports' / 'objective3_demonstration'
OUT.mkdir(parents=True, exist_ok=True)
print('Environnement prêt.')

Environnement prêt.


## 1. Les données IoT en élevage laitier

**Capteur IceTag** (accéléromètre fixé à la patte) : enregistre, minute par minute, le nombre de
pas, l'indice de mouvement, le temps debout/couché et les transitions. Agrégé ici en
**intervalles de 15 minutes**.

**Capteur HOBO** (environnemental) : température et humidité toutes les 5 minutes, d'où l'on
calcule le **THI** (Temperature-Humidity Index), indicateur de stress thermique.

Affichons un échantillon de données IceTag prétraitées.

In [2]:
act = pd.read_csv(O1 / 'summer_2019_pipeline_input_15min.csv')
act['Start'] = pd.to_datetime(act['Start'])
print(f"IceTag Summer 2019 : {len(act):,} intervalles de 15 min, {act['Cow'].nunique()} vaches")
print(f"Période : {act['Start'].min().date()} → {act['Start'].max().date()}")
act[['Cow', 'Start', 'Steps', 'Motion Index', 'Lying Time', 'Transitions']].head()

IceTag Summer 2019 : 136,726 intervalles de 15 min, 18 vaches
Période : 2019-06-05 → 2019-09-06


,Cow,Start,Steps,Motion Index,Lying Time,Transitions
0,2062,2019-06-05 13:00:00,10,27,0:04:23,2
1,2062,2019-06-05 13:15:00,9,20,0:00:12,5
2,2062,2019-06-05 13:30:00,9,17,0:00:12,2
3,2062,2019-06-05 13:45:00,16,32,0:00:38,9
4,2062,2019-06-05 14:00:00,9,19,0:00:00,0


### Profil d'activité journalier d'une vache

Le signal IoT révèle le rythme quotidien : pics d'activité, périodes de repos.

In [3]:
cow0 = act['Cow'].iloc[0]
one = act[act['Cow'] == cow0].copy()
one['hour'] = one['Start'].dt.hour
prof = one.groupby('hour')['Steps'].mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prof.index, prof.values, marker='o', color='#2c7fb8')
ax.fill_between(prof.index, prof.values, alpha=0.2, color='#2c7fb8')
ax.set_title(f'Profil journalier moyen — vache {cow0} (Summer 2019)')
ax.set_xlabel('Heure de la journée'); ax.set_ylabel('Pas moyens / 15 min')
fig.tight_layout(); fig.savefig(OUT / 'demo_profil_journalier.png', dpi=120)
plt.show()

## 2. Détection d'anomalies comportementales (pipeline boiterie)

Le pipeline (Isolation Forest + règles métier) compare chaque vache à sa propre ligne de base et
émet une **alerte** quand le comportement dévie fortement. Voici la synthèse des 4 essais.

In [4]:
ms = pd.read_csv(O1 / 'objective1_multi_season_summary.csv')
view = ms[['season', 'n_cows', 'n_intervals', 'lameness_notifs', 'notifs_per_100_cow_days']].copy()
view.columns = ['Essai', 'Vaches', 'Intervalles 15 min', 'Alertes', 'Alertes/100 cow-jours']
print(view.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(view['Essai'], view['Alertes/100 cow-jours'], color='#d95f0e')
ax.set_title('Taux d\'alertes normalisé par essai')
ax.set_ylabel('Alertes / 100 cow-jours')
fig.tight_layout(); fig.savefig(OUT / 'demo_alertes_par_essai.png', dpi=120)
plt.show()

      Essai  Vaches  Intervalles 15 min  Alertes  Alertes/100 cow-jours
  fall_2019      30               93860      105                  10.74
  fall_2021      10                5131        4                   7.48
summer_2019      18              139111      127                   8.76
winter_2019      17              136929      149                  10.45


### Quand les alertes surviennent-elles ?

Distribution horaire des alertes — elles se concentrent en journée, périodes d'activité.

In [5]:
al = pd.read_csv(O1 / 'summer_2019_pipeline_alerts_only.csv')
al['T'] = pd.to_datetime(al['T'])
byh = al['T'].dt.hour.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(byh.index, byh.values, color='#756bb1')
ax.set_title('Distribution horaire des alertes (Summer 2019)')
ax.set_xlabel('Heure'); ax.set_ylabel('Nombre d\'alertes')
fig.tight_layout(); fig.savefig(OUT / 'demo_alertes_horaire.png', dpi=120)
plt.show()
print(f"Confiance moyenne des alertes : {al['lame_confidence'].mean():.0f}%")

Confiance moyenne des alertes : 47%


## 3. Environnement × comportement

En croisant l'activité IceTag avec le THI, on observe comment les conditions thermiques
influencent le comportement des vaches.

In [6]:
env = pd.read_csv(O2 / 'summer2019_icetag_environnement_15min.csv')
env['Start'] = pd.to_datetime(env['Start'])
env['hour'] = env['Start'].dt.hour
day = env[(env['hour'] >= 6) & (env['hour'] < 20)].copy()

def thi_cat(t):
    return 'Aucun\n(<68)' if t < 68 else 'Léger\n(68-72)' if t < 72 else 'Modéré\n(72-80)' if t < 80 else 'Sévère\n(≥80)'
day['THI_cat'] = day['THI'].apply(thi_cat)
order = ['Aucun\n(<68)', 'Léger\n(68-72)', 'Modéré\n(72-80)', 'Sévère\n(≥80)']
act_by = day.groupby('THI_cat')['Steps'].mean().reindex([o for o in order if o in day['THI_cat'].unique()])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(act_by.index, act_by.values, color='#e34a33')
ax.set_title('Activité moyenne selon le niveau de stress thermique')
ax.set_xlabel('Niveau de stress thermique (THI)'); ax.set_ylabel('Pas moyens / 15 min')
fig.tight_layout(); fig.savefig(OUT / 'demo_activite_THI.png', dpi=120)
plt.show()

In [7]:
# Composition comportementale vs THI (au niveau jour — unité correcte)
be = pd.read_csv(O2 / 'summer2019_comportement_environnement.csv')
behav = ['Pct_eating', 'Pct_locomotion', 'Pct_Idle', 'Pct_lying']
daily = be.groupby('jour').agg(THI=('THI_jour', 'first'),
                               **{b: (b, 'mean') for b in behav}).dropna()
fig, ax = plt.subplots(figsize=(8, 4.5))
for b in behav:
    ax.scatter(daily['THI'], daily[b] * 100, label=b.replace('Pct_', ''), s=50, alpha=0.7)
ax.set_title('Composition comportementale vs THI (par jour, Summer 2019)')
ax.set_xlabel('THI journalier'); ax.set_ylabel('% du temps observé'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / 'demo_comportement_THI.png', dpi=120)
plt.show()
print(f"(Analyse sur {len(daily)} jours de scan — résultat suggestif, faible puissance.)")

(Analyse sur 0 jours de scan — résultat suggestif, faible puissance.)


## 4. Guide d'interprétation

**Comment lire une alerte du pipeline ?**
Une alerte signale que le comportement d'une vache s'écarte fortement de son habitude. C'est un
signal de **dépistage** (« cette vache mérite un coup d'œil »), pas un diagnostic. Les causes
possibles : boiterie sévère, maladie aiguë, chaleurs (œstrus), vêlage, ou événement de gestion.

**Ce que le pipeline détecte bien :** les changements d'activité marqués et soutenus.

**Ce qu'il ne détecte pas :** la boiterie légère, qui ne modifie pas la quantité de mouvement —
elle se voit dans l'asymétrie de démarche, que l'IceTag ne mesure pas.

**Environnement :** l'activité et certains comportements (alimentation) varient avec le THI, ce
qui montre l'intérêt de croiser capteurs d'activité et capteurs environnementaux.

**Bonnes pratiques pour ce type de données IoT :**
- Toujours normaliser le taux d'alertes par cow-jour (comparer des essais de durées différentes).
- Contrôler les confondants (expérience, heure de la journée) avant d'interpréter une relation.
- Traiter le bon niveau d'analyse (le jour pour l'environnement, la vache pour le clinique).
- Distinguer un signal de dépistage d'un diagnostic clinique.

In [8]:
summary = f'''DÉMONSTRATION IoT — bovins laitiers (synthèse)

Données       : 4 essais, capteurs IceTag (activité) + HOBO (environnement)
Pipeline      : Isolation Forest + règles métier, intervalles de 15 min
Alertes       : {int(ms["lameness_notifs"].sum())} au total sur les 4 essais
Usage         : dépistage d'anomalies comportementales (santé, chaleurs, vêlage)
Limite clé    : boiterie légère non détectable par capteur d'activité
Environnement : activité et alimentation liées au THI (effet thermique)

Figures générées dans : {OUT}
'''
(OUT / 'synthese_demonstration.txt').write_text(summary, encoding='utf-8')
print(summary)

DÉMONSTRATION IoT — bovins laitiers (synthèse)

Données       : 4 essais, capteurs IceTag (activité) + HOBO (environnement)
Pipeline      : Isolation Forest + règles métier, intervalles de 15 min
Alertes       : 385 au total sur les 4 essais
Usage         : dépistage d'anomalies comportementales (santé, chaleurs, vêlage)
Limite clé    : boiterie légère non détectable par capteur d'activité
Environnement : activité et alimentation liées au THI (effet thermique)

Figures générées dans : /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective3_demonstration

